In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
import kagglehub

# Download the Movies dataset from Kaggle (cached locally by KaggleHub)
# Switch to another dataset slug if you prefer (e.g., 'grouplens/movielens-20m-dataset')
DATASET_SLUG = "rounakbanik/the-movies-dataset"
path = kagglehub.dataset_download(DATASET_SLUG)
print("Path to dataset files:", path)

# Resolve files present in the dataset (prefer the small variants for quick runs)
movies_candidates = [
    os.path.join(path, "movies.csv"),
    os.path.join(path, "movies_metadata.csv")
]
ratings_candidates = [
    os.path.join(path, "ratings_small.csv"),
    os.path.join(path, "ratings.csv")
]
links_candidates = [
    os.path.join(path, "links_small.csv"),
    os.path.join(path, "links.csv")
]

def pick_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

movies_fp = pick_existing(movies_candidates)
ratings_fp = pick_existing(ratings_candidates)
links_fp = pick_existing(links_candidates)

if ratings_fp is None:
    raise FileNotFoundError("Couldn't find ratings.csv or ratings_small.csv in the dataset")

# Load ratings first (MovieLens schema: userId,movieId,rating,timestamp)
ratings = pd.read_csv(ratings_fp)

# Build a movies DataFrame with columns [movieId,title,genres]
if movies_fp and movies_fp.endswith("movies.csv"):
    # Direct MovieLens format
    movies = pd.read_csv(movies_fp)
    if "title" in movies.columns:
        movies['title'] = movies['title'].astype(str).str.lower()
elif movies_fp and movies_fp.endswith("movies_metadata.csv") and links_fp:
    # Map MovieLens movieId -> TMDB id via links, then join to movies_metadata for titles/genres
    links = pd.read_csv(links_fp)
    mm = pd.read_csv(movies_fp, low_memory=False)
    # Keep valid numeric ids
    mm = mm[pd.to_numeric(mm['id'], errors='coerce').notna()].copy()
    mm['id'] = mm['id'].astype(int)
    links = links[pd.to_numeric(links['tmdbId'], errors='coerce').notna()].copy()
    links['tmdbId'] = links['tmdbId'].astype(int)
    merged = links.merge(mm[["id", "title", "genres"]], left_on="tmdbId", right_on="id", how="left")
    movies = merged[["movieId", "title", "genres"]].copy()
    movies['title'] = movies['title'].astype(str).str.lower()
    # Convert genres JSON list string to pipe-delimited string
    def genres_to_pipe(g):
        try:
            items = ast.literal_eval(g) if isinstance(g, str) else []
            names = [d.get('name') for d in items if isinstance(d, dict) and 'name' in d]
            return "|".join(names)
        except Exception:
            return ""
    if 'genres' in movies.columns:
        movies['genres'] = movies['genres'].apply(genres_to_pipe)
else:
    raise FileNotFoundError("Couldn't find movies.csv or a valid movies_metadata+links pair in the dataset")


In [2]:
movies['title'] = movies['title'].str.lower()


In [3]:
movies.head()


In [4]:
ratings.head()


In [5]:
final_dataset = ratings.pivot(
    index='movieId', columns='userId', values='rating')
final_dataset.head()


In [ ]:
final_dataset.fillna(0, inplace=True)
final_dataset.head()


In [ ]:
no_user_voted = ratings.groupby('movieId')['rating'].agg('count')
no_movies_voted = ratings.groupby('userId')['rating'].agg('count')


In [ ]:
f, ax = plt.subplots(1, 1, figsize=(16, 4))
# ratings['rating'].plot(kind='hist')
plt.scatter(no_user_voted.index, no_user_voted, color='mediumseagreen')
plt.axhline(y=10, color='r')
plt.xlabel('MovieId')
plt.ylabel('No. of users voted')
plt.show()


In [ ]:
final_dataset = final_dataset.loc[:,
                                  no_movies_voted[no_movies_voted > 50].index]
final_dataset


In [ ]:
sample = np.array([[0, 0, 3, 0, 0], [4, 0, 0, 0, 2], [0, 0, 0, 0, 1]])
sparsity = 1.0 - (np.count_nonzero(sample) / float(sample.size))
print(sparsity)


In [ ]:
csr_sample = csr_matrix(sample)
print(csr_sample)


In [ ]:
csr_data = csr_matrix(final_dataset.values)
final_dataset.reset_index(inplace=True)


In [ ]:
knn = NearestNeighbors(metric='cosine', algorithm='brute',
                       n_neighbors=20, n_jobs=-1)
knn.fit(csr_data)


In [ ]:
def get_movie_recommendation(movie_name):
    n_movies_to_reccomend = 10
    movie_list = movies[movies['title'].str.contains(movie_name)]
    if len(movie_list):
        movie_idx = movie_list.iloc[0]['movieId']
        movie_idx = final_dataset[final_dataset['movieId']
                                  == movie_idx].index[0]
        distances, indices = knn.kneighbors(
            csr_data[movie_idx], n_neighbors=n_movies_to_reccomend+1)
        rec_movie_indices = sorted(list(zip(indices.squeeze().tolist(
        ), distances.squeeze().tolist())), key=lambda x: x[1])[:0:-1]
        recommend_frame = []
        for val in rec_movie_indices:
            movie_idx = final_dataset.iloc[val[0]]['movieId']
            idx = movies[movies['movieId'] == movie_idx].index
            recommend_frame.append(
                {'Title': movies.iloc[idx]['title'].values[0], 'Distance': val[1]})
        df = pd.DataFrame(recommend_frame, index=range(
            1, n_movies_to_reccomend+1))
        return df
    else:
        return "No movies found. Please check your input"


In [ ]:
get_movie_recommendation('aladdin')


In [ ]:
message ='helloworld'
